# MRIxFields2026 Task 3 — conditional FPS-Former

This notebook orchestrates the source-aware Task 3 adaptation in `task3_fpsformer/`; it does not copy the authors' reconstruction-only model into notebook cells. The workflow is: stage code and paired prospective data into `/content`, unmount Drive, audit and cache the data, run tests, run a tiny smoke, train a chunked and resumable subject-held-out paid run, optionally continue on all subjects, infer 180 validation volumes, use the current official builder, validate `task3.zip`, and optionally export selected artifacts.

Every expensive or externally changing action defaults to **off**. Enable one switch at a time. The notebook never uploads a submission.


## 1. Mount Drive, configure paths, and stage the Python package

`DRIVE_DATA_ROOT_VALUE` must be the dataset root containing `training_prospective` or `Training_prospective`. Do not set it to the validation split itself. Browser upload is the default for the package ZIP; switch it off only when the ZIP is already in Drive.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

DRIVE_MOUNT = Path('/content/drive')
drive.mount(str(DRIVE_MOUNT))

RUNTIME_ROOT_VALUE = '/content/mrixfields_task3/fpsformer'  # @param {type:'string'}
DRIVE_DATA_ROOT_VALUE = '/content/drive/MyDrive/mrixfields2026'  # @param {type:'string'}
DRIVE_CODE_ZIP_VALUE = '/content/drive/MyDrive/mrixfields_task3_source/task3_fpsformer_colab.zip'  # @param {type:'string'}
DRIVE_PAPER_PDF_VALUE = ''  # @param {type:'string'}
DRIVE_INITIAL_CHECKPOINT_VALUE = ''  # @param {type:'string'}
DRIVE_EXPORT_ROOT_VALUE = '/content/drive/MyDrive/mrixfields_task3_exports/fpsformer'  # @param {type:'string'}
USE_BROWSER_UPLOAD_FOR_CODE = True  # @param {type:'boolean'}

RUNTIME_ROOT = Path(RUNTIME_ROOT_VALUE).expanduser().resolve()
PROJECT_DIR = RUNTIME_ROOT / 'project'
DATA_DIR = RUNTIME_ROOT / 'data'
PREPROCESSED_DIR = RUNTIME_ROOT / 'preprocessed' / 'fpsformer'
OUTPUT_DIR = RUNTIME_ROOT / 'outputs'
INFERENCE_DIR = RUNTIME_ROOT / 'inference'
SUBMISSION_DIR = RUNTIME_ROOT / 'submission'
RESULTS_DIR = RUNTIME_ROOT / 'results'
REPORT_DIR = RESULTS_DIR / 'reports'
STAGED_CHECKPOINT_DIR = RUNTIME_ROOT / 'staged_checkpoints'
OFFICIAL_REPO = RUNTIME_ROOT / 'official' / 'MRIxFields2026'
DRIVE_DATA_ROOT = Path(DRIVE_DATA_ROOT_VALUE).expanduser()

drive_root = DRIVE_MOUNT.resolve()
if RUNTIME_ROOT == drive_root or drive_root in RUNTIME_ROOT.parents:
    raise ValueError('RUNTIME_ROOT must be under /content, not mounted Drive.')
for path in (PROJECT_DIR, DATA_DIR, PREPROCESSED_DIR, OUTPUT_DIR, INFERENCE_DIR, SUBMISSION_DIR, RESULTS_DIR, REPORT_DIR, STAGED_CHECKPOINT_DIR, OFFICIAL_REPO.parent):
    path.mkdir(parents=True, exist_ok=True)

RUNTIME_WORK_PATHS = (PROJECT_DIR, DATA_DIR, PREPROCESSED_DIR, OUTPUT_DIR, INFERENCE_DIR, SUBMISSION_DIR, RESULTS_DIR, OFFICIAL_REPO)

def drive_is_mounted():
    return os.path.ismount(DRIVE_MOUNT) or (DRIVE_MOUNT / 'MyDrive').is_dir()

def assert_runtime_only():
    if drive_is_mounted():
        raise RuntimeError('Drive must be unmounted before audits, caching, training, inference, or packaging.')
    for path in RUNTIME_WORK_PATHS:
        resolved = Path(path).expanduser().resolve()
        if resolved == drive_root or drive_root in resolved.parents:
            raise RuntimeError(f'Runtime path points to Drive: {resolved}')

if USE_BROWSER_UPLOAD_FOR_CODE:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if len(zip_names) != 1:
        raise RuntimeError(f'Upload exactly one package zip; received {list(uploaded)}')
    code_zip_source = Path('/content') / zip_names[0]
else:
    code_zip_source = Path(DRIVE_CODE_ZIP_VALUE).expanduser()
if not code_zip_source.is_file():
    raise FileNotFoundError(f'Package zip not found: {code_zip_source}')
runtime_zip = RUNTIME_ROOT / 'task3_fpsformer_colab.zip'
shutil.copy2(code_zip_source, runtime_zip)
with zipfile.ZipFile(runtime_zip) as archive:
    members = archive.namelist()
    unsafe = [name for name in members if Path(name).is_absolute() or '..' in Path(name).parts]
    if unsafe:
        raise RuntimeError(f'Unsafe zip members: {unsafe[:5]}')
    archive.extractall(PROJECT_DIR)
package_dir = PROJECT_DIR / 'task3_fpsformer'
for required in ('__init__.py', 'data.py', 'model.py', 'train.py', 'infer.py', 'requirements-colab.txt'):
    if not (package_dir / required).is_file():
        raise FileNotFoundError(f'Package zip is missing task3_fpsformer/{required}')
paper_source = Path(DRIVE_PAPER_PDF_VALUE).expanduser() if DRIVE_PAPER_PDF_VALUE.strip() else None
if paper_source is not None:
    if not paper_source.is_file():
        raise FileNotFoundError(paper_source)
    shutil.copy2(paper_source, PROJECT_DIR / '2412.10776v1.pdf')
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.environ['TORCH_HOME'] = str(RUNTIME_ROOT / '.cache' / 'torch')
print('Runtime project:', PROJECT_DIR)
print('Package:', package_dir)
print('Dataset root selected for staging:', DRIVE_DATA_ROOT)


## 2. Install dependencies and verify the Colab GPU

Colab already supplies its CUDA-matched PyTorch build; the package requirements do not replace it.


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package_dir / 'requirements-colab.txt')])
import nibabel as nib
import numpy as np
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU before continuing.')
properties = torch.cuda.get_device_properties(0)
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', properties.name, round(properties.total_memory / 2**30, 1), 'GiB')


## 3. Stage the paired training and validation splits, then unmount Drive

Enable both staging switches on a fresh runtime if disk space allows. Only the exact Task 3 split trees are copied. The source dataset is never renamed or modified. A saved initialization checkpoint is optional.


In [ ]:
STAGE_TRAINING_DATA = False  # @param {type:'boolean'}
STAGE_VALIDATION_DATA = False  # @param {type:'boolean'}
STAGE_INITIAL_CHECKPOINT = False  # @param {type:'boolean'}

def resolve_source_split(root, aliases, label):
    root = Path(root).expanduser()
    matches = [root / alias for alias in aliases if (root / alias).is_dir()]
    if len(matches) != 1:
        raise RuntimeError(f'{label}: expected exactly one of {aliases} below dataset root {root}; found {matches}. Set DRIVE_DATA_ROOT_VALUE to the parent mrixfields2026 directory, not a split directory.')
    return matches[0]

def stage_nifti_split(source, destination):
    source = Path(source).expanduser().resolve()
    destination = Path(destination).expanduser().resolve()
    source_files = sorted(source.rglob('*.nii.gz'))
    if not source_files:
        raise RuntimeError(f'No NIfTI files in {source}')
    inventory = {path.relative_to(source): path.stat().st_size for path in source_files}
    required_bytes = sum(inventory.values())
    free_bytes = shutil.disk_usage(RUNTIME_ROOT).free
    if free_bytes < required_bytes + 5 * 2**30:
        raise RuntimeError(f'Insufficient runtime disk: need source bytes plus 5 GiB margin ({required_bytes}), free={free_bytes}')
    destination.mkdir(parents=True, exist_ok=True)
    for index, (relative, size) in enumerate(inventory.items(), 1):
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.stat().st_size == size:
            continue
        temporary = target.with_name(f'.{target.name}.part')
        shutil.copy2(source / relative, temporary)
        if temporary.stat().st_size != size:
            raise RuntimeError(f'Incomplete copy: {temporary}')
        os.replace(temporary, target)
        if index % 10 == 0 or index == len(inventory):
            print(f'Staged {index}/{len(inventory)} files into {destination}')
    runtime_inventory = {path.relative_to(destination): path.stat().st_size for path in destination.rglob('*.nii.gz')}
    if runtime_inventory != inventory:
        raise RuntimeError(f'Runtime inventory differs from source: {destination}')

if STAGE_TRAINING_DATA:
    training_source = resolve_source_split(DRIVE_DATA_ROOT, ('Training_prospective', 'training_prospective'), 'paired training split')
    stage_nifti_split(training_source, DATA_DIR / 'training_prospective')
else:
    print('Training-data staging is OFF.')
if STAGE_VALIDATION_DATA:
    validation_source = resolve_source_split(DRIVE_DATA_ROOT, ('Validating_prospective', 'validation_prospective', 'validating_prospective'), 'validation split')
    stage_nifti_split(validation_source, DATA_DIR / 'validation_prospective')
else:
    print('Validation-data staging is OFF.')
if STAGE_INITIAL_CHECKPOINT:
    checkpoint_source = Path(DRIVE_INITIAL_CHECKPOINT_VALUE).expanduser()
    if not checkpoint_source.is_file():
        raise FileNotFoundError(checkpoint_source)
    staged_initial_checkpoint = STAGED_CHECKPOINT_DIR / checkpoint_source.name
    shutil.copy2(checkpoint_source, staged_initial_checkpoint)
    print('Staged checkpoint:', staged_initial_checkpoint)
subprocess.run(['sync'], check=True)
drive.flush_and_unmount()
assert_runtime_only()
print('Runtime staging complete. Drive is unmounted; all processing is runtime-only.')


## 4. Run functional package tests

The notebook's static structure is tested before the ZIP is built; Colab runs the model/data tests from the staged package.


In [ ]:
RUN_PACKAGE_TESTS = True  # @param {type:'boolean'}
assert_runtime_only()
if RUN_PACKAGE_TESTS:
    subprocess.check_call([sys.executable, '-m', 'compileall', '-q', 'task3_fpsformer'])
    test_result = subprocess.run([sys.executable, '-m', 'unittest', '-v', 'task3_fpsformer.tests.test_fpsformer'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(test_result.stdout)
    if test_result.returncode:
        raise RuntimeError(f'Functional package tests failed with exit code {test_result.returncode}.')
else:
    print('Package tests are OFF.')


## 5. Audit the paired prospective training volumes

This must pass before cache creation. It checks all 15 domains, exact prospective subject pairing, full geometry, float32 dtype, finite values, and `[0,1]` range.


In [ ]:
RUN_RAW_AUDIT = False  # @param {type:'boolean'}
assert_runtime_only()
if RUN_RAW_AUDIT:
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.data', 'audit-raw', '--data-dir', str(DATA_DIR), '--check-values', '--report', str(REPORT_DIR / 'raw_audit.json')])
else:
    print('Raw training-data audit is OFF.')


## 6. Build and audit the 220-slice paired cache

Cache creation writes only under runtime storage. It extracts axial indices `[72,292)` and is resumable.


In [ ]:
RUN_CACHE_PREPARATION = False  # @param {type:'boolean'}
assert_runtime_only()
if RUN_CACHE_PREPARATION:
    if not (REPORT_DIR / 'raw_audit.json').is_file():
        raise RuntimeError('Run and pass the raw audit first.')
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.data', 'prepare-cache', '--data-dir', str(DATA_DIR), '--cache-dir', str(PREPROCESSED_DIR)])
else:
    print('Cache preparation is OFF.')

RUN_CACHE_AUDIT = False  # @param {type:'boolean'}
if RUN_CACHE_AUDIT:
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.data', 'audit-cache', '--cache-dir', str(PREPROCESSED_DIR), '--report', str(REPORT_DIR / 'cache_audit.json')])
else:
    print('Cache audit is OFF.')


## 7. Two-update tiny smoke training

This tests the complete data/model/loss/checkpoint/holdout path. It is not a quality result. Set the baseline acknowledgement only because your organizer-pretrained 180-case milestone has already completed. The smoke deliberately uses `--workers 0` so a Colab multiprocessing/shared-memory failure cannot hide the model result.


In [ ]:
BASELINE_MILESTONE_COMPLETE = False  # @param {type:'boolean'}
RUN_TINY_SMOKE = False  # @param {type:'boolean'}
TINY_SMOKE_DIR = OUTPUT_DIR / 'tiny_smoke'
assert_runtime_only()
if RUN_TINY_SMOKE:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Acknowledge the completed baseline milestone before training.')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is no longer available; reconnect to a Colab GPU runtime.')
    cache_audit_path = REPORT_DIR / 'cache_audit.json'
    if not cache_audit_path.is_file():
        raise RuntimeError('Run and pass the cache audit first.')
    cache_audit = json.loads(cache_audit_path.read_text())
    complete_subjects = cache_audit.get('complete_subjects', [])
    if cache_audit.get('status') != 'passed' or len(complete_subjects) < 2:
        raise RuntimeError(f'Tiny holdout smoke requires a passed cache and at least two complete subjects; status={cache_audit.get("status")}, subjects={complete_subjects}')
    existing_smoke_checkpoints = sorted((TINY_SMOKE_DIR / 'checkpoints').glob('*.pt'))
    if existing_smoke_checkpoints:
        raise RuntimeError(f'Tiny smoke already has checkpoints; it passed previously or needs an explicit fresh output directory: {existing_smoke_checkpoints}')
    print('Tiny smoke preflight passed. Complete subjects:', complete_subjects)
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.train', '--cache-dir', str(PREPROCESSED_DIR), '--output-dir', str(TINY_SMOKE_DIR), '--preset', 'tiny', '--device', 'cuda', '--amp', 'fp16', '--crop-size', '128', '--eval-crop-size', '128', '--epochs', '1', '--updates-per-epoch', '2', '--val-samples-per-pair', '1', '--workers', '0', '--progress', '--baseline-milestone-complete'])
else:
    print('Tiny smoke training is OFF.')


## 8. Paid-Colab subject-held-out training

The A100 recipe uses batch 16, 256-pixel random crops, paired left-right flips, EMA checkpoint selection, and full-slice holdout evaluation. Training runs in 45-minute chunks because paid Colab GPU type and runtime lifetime are not guaranteed. Each chunk stops at an epoch boundary, writes an atomic local checkpoint, then optionally mounts Drive only while training is stopped to update verified recovery copies. `initialize` starts a fresh optimizer schedule from a compatible held-out model; `resume` is only for an interrupted paid-run `last.pt`. Public-validation targets remain private, so the reported values are prospective holdout slice proxies.


In [ ]:
RUN_PAID_TRAINING = False  # @param {type:'boolean'}
PAID_RUN_NAME = 'a100_holdout_ema_01'  # @param {type:'string'}
PAID_START_MODE = 'scratch'  # @param ['scratch', 'initialize', 'resume']
PAID_CHECKPOINT_VALUE = ''  # @param {type:'string'}
PAID_TARGET_EPOCHS = 100  # @param {type:'integer'}
PAID_UPDATES_PER_EPOCH = 1000  # @param {type:'integer'}
PAID_CHUNK_MINUTES = 45  # @param {type:'integer'}
PAID_MAX_CHUNKS = 12  # @param {type:'integer'}
PAID_BATCH_SIZE = 16  # @param {type:'integer'}
PAID_EVAL_BATCH_SIZE = 4  # @param {type:'integer'}
PAID_CROP_SIZE = 256  # @param {type:'integer'}
PAID_EVAL_CROP_SIZE = 0  # 0 evaluates complete 364x436 slices
PAID_VAL_SAMPLES_PER_PAIR = 30  # all scored slices [150,180)
PAID_VAL_SLICE_START = 150
PAID_VAL_SLICE_END = 180
PAID_EVAL_EVERY = 5  # @param {type:'integer'}
PAID_WORKERS = 4  # @param {type:'integer'}
PAID_EMA_BETA = 0.999  # @param {type:'number'}
PAID_LEFT_RIGHT_FLIP_PROBABILITY = 0.5  # @param {type:'number'}
PAID_LOSS_LPIPS = 0.1  # @param {type:'number'}
PAID_BACKUP_TO_DRIVE = True  # @param {type:'boolean'}
EVAL_LPIPS = True  # @param {type:'boolean'}
PAID_RUN_DIR = OUTPUT_DIR / PAID_RUN_NAME
PAID_BACKUP_DIR = Path(DRIVE_EXPORT_ROOT_VALUE).expanduser() / 'paid_backups' / PAID_RUN_NAME

def sha256_stream(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def backup_paid_run(epoch, global_step):
    if not PAID_BACKUP_TO_DRIVE:
        print('Paid-run Drive backup is OFF.')
        return
    if drive_is_mounted():
        raise RuntimeError('Drive unexpectedly mounted before paid-run backup.')
    drive.mount(str(DRIVE_MOUNT))
    records = []
    try:
        PAID_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        sources = [(PAID_RUN_DIR / 'checkpoints' / 'last.pt', 'last.pt'), (PAID_RUN_DIR / 'checkpoints' / 'best.pt', 'best.pt'), (PAID_RUN_DIR / 'provenance.json', 'provenance.json'), (PAID_RUN_DIR / 'metrics.jsonl', f'metrics_through_epoch_{epoch:04d}.jsonl')]
        for source, name in sources:
            if not source.is_file():
                continue
            source_hash = sha256_stream(source)
            destination = PAID_BACKUP_DIR / name
            if destination.is_file() and destination.stat().st_size == source.stat().st_size and sha256_stream(destination) == source_hash:
                records.append({'path': str(destination), 'size': source.stat().st_size, 'sha256': source_hash, 'reused': True})
                continue
            temporary = destination.with_name(f'.{destination.name}.part')
            temporary.unlink(missing_ok=True)
            shutil.copy2(source, temporary)
            if temporary.stat().st_size != source.stat().st_size or sha256_stream(temporary) != source_hash:
                raise RuntimeError(f'Paid-run backup verification failed: {destination}')
            os.replace(temporary, destination)
            records.append({'path': str(destination), 'size': source.stat().st_size, 'sha256': source_hash, 'reused': False})
        manifest = {'run_name': PAID_RUN_NAME, 'epoch': epoch, 'global_step': global_step, 'target_epochs': PAID_TARGET_EPOCHS, 'updates_per_epoch': PAID_UPDATES_PER_EPOCH, 'records': records}
        manifest_temporary = PAID_BACKUP_DIR / '.backup_manifest.json.part'
        manifest_temporary.write_text(json.dumps(manifest, indent=2) + '\n')
        os.replace(manifest_temporary, PAID_BACKUP_DIR / 'backup_manifest.json')
        print('Verified paid-run backup:', PAID_BACKUP_DIR)
    finally:
        drive.flush_and_unmount()

assert_runtime_only()
if RUN_PAID_TRAINING:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Acknowledge the completed baseline milestone before training.')
    if PAID_START_MODE not in {'scratch', 'initialize', 'resume'}:
        raise ValueError(f'Unknown PAID_START_MODE: {PAID_START_MODE}')
    if min(PAID_TARGET_EPOCHS, PAID_UPDATES_PER_EPOCH, PAID_CHUNK_MINUTES, PAID_MAX_CHUNKS, PAID_BATCH_SIZE, PAID_EVAL_BATCH_SIZE, PAID_CROP_SIZE, PAID_VAL_SAMPLES_PER_PAIR, PAID_EVAL_EVERY) < 1:
        raise ValueError('Paid-run epochs, updates, batch sizes, crop, and evaluation settings must be positive.')
    if not 0 <= PAID_LEFT_RIGHT_FLIP_PROBABILITY <= 1 or not 0 <= PAID_EMA_BETA < 1 or PAID_LOSS_LPIPS < 0:
        raise ValueError('Invalid augmentation, EMA, or loss setting.')
    cache_audit = json.loads((REPORT_DIR / 'cache_audit.json').read_text())
    complete_subjects = cache_audit.get('complete_subjects', [])
    if cache_audit.get('status') != 'passed' or len(complete_subjects) < 2:
        raise RuntimeError('Paid training requires the passed paired cache audit.')
    expected_holdout = complete_subjects[-1]
    staged_candidates = [Path(PAID_CHECKPOINT_VALUE).expanduser()] if PAID_CHECKPOINT_VALUE.strip() else sorted(STAGED_CHECKPOINT_DIR.glob('*.pt')) + sorted(STAGED_CHECKPOINT_DIR.glob('*.pth'))
    start_checkpoint = None
    if PAID_START_MODE != 'scratch':
        if len(staged_candidates) != 1 or not staged_candidates[0].is_file():
            raise RuntimeError(f'{PAID_START_MODE} requires exactly one staged or explicit checkpoint; found {staged_candidates}')
        start_checkpoint = staged_candidates[0]
        try:
            checkpoint_metadata = torch.load(start_checkpoint, map_location='cpu', weights_only=True)
        except TypeError:
            checkpoint_metadata = torch.load(start_checkpoint, map_location='cpu')
        if checkpoint_metadata.get('format') != 'task3_conditional_fpsformer_v1' or checkpoint_metadata.get('holdout_subject') != expected_holdout:
            raise RuntimeError(f'Checkpoint is not a compatible held-out Task 3 run for {expected_holdout}: {start_checkpoint}')
        del checkpoint_metadata
    print('Paid run:', PAID_RUN_NAME, 'GPU:', properties.name, 'target updates:', PAID_TARGET_EPOCHS * PAID_UPDATES_PER_EPOCH, 'free runtime GiB:', round(shutil.disk_usage(RUNTIME_ROOT).free / 2**30, 1))
    for chunk_index in range(1, PAID_MAX_CHUNKS + 1):
        assert_runtime_only()
        local_last = PAID_RUN_DIR / 'checkpoints' / 'last.pt'
        command = [sys.executable, '-m', 'task3_fpsformer.train', '--cache-dir', str(PREPROCESSED_DIR), '--output-dir', str(PAID_RUN_DIR), '--preset', 'portable', '--device', 'cuda', '--amp', 'bf16', '--batch-size', str(PAID_BATCH_SIZE), '--eval-batch-size', str(PAID_EVAL_BATCH_SIZE), '--gradient-accumulation', '1', '--crop-size', str(PAID_CROP_SIZE), '--eval-crop-size', str(PAID_EVAL_CROP_SIZE), '--epochs', str(PAID_TARGET_EPOCHS), '--updates-per-epoch', str(PAID_UPDATES_PER_EPOCH), '--max-runtime-minutes', str(PAID_CHUNK_MINUTES), '--val-samples-per-pair', str(PAID_VAL_SAMPLES_PER_PAIR), '--val-slice-start', str(PAID_VAL_SLICE_START), '--val-slice-end', str(PAID_VAL_SLICE_END), '--eval-every', str(PAID_EVAL_EVERY), '--workers', str(PAID_WORKERS), '--ema-beta', str(PAID_EMA_BETA), '--left-right-flip-probability', str(PAID_LEFT_RIGHT_FLIP_PROBABILITY), '--loss-lpips', str(PAID_LOSS_LPIPS), '--progress', '--baseline-milestone-complete']
        if local_last.is_file():
            command.extend(['--resume', str(local_last)])
        elif PAID_START_MODE == 'initialize':
            command.extend(['--initialize', str(start_checkpoint)])
        elif PAID_START_MODE == 'resume':
            command.extend(['--resume', str(start_checkpoint)])
        if EVAL_LPIPS:
            command.append('--eval-lpips')
        print(f'Paid training chunk {chunk_index}/{PAID_MAX_CHUNKS}')
        subprocess.check_call(command)
        try:
            saved = torch.load(local_last, map_location='cpu', weights_only=True)
        except TypeError:
            saved = torch.load(local_last, map_location='cpu')
        completed_epoch = int(saved['epoch'])
        completed_step = int(saved['global_step'])
        del saved
        backup_paid_run(completed_epoch, completed_step)
        if completed_epoch >= PAID_TARGET_EPOCHS:
            print('Paid training target complete.')
            break
    else:
        print('Paid chunk limit reached. Resume from the verified Drive last.pt in a new runtime.')
else:
    print('Paid Colab training is OFF.')


## 9. Review holdout metrics and choose a checkpoint

For nRMSE and LPIPS lower is better; for SSIM higher is better. Do not select on public validation repeatedly before the prospective holdout experiment is understood.


In [ ]:
assert_runtime_only()
metrics_path = PAID_RUN_DIR / 'metrics.jsonl'
if metrics_path.is_file():
    metric_records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    for record in metric_records:
        print('epoch', record['epoch'], 'step', record['global_step'], json.dumps(record.get('holdout', {}), indent=2))
else:
    print('Run paid held-out training first:', metrics_path)


## 10. Optional all-subject continuation

After selecting settings on the held-out pilot, initialize a fresh run with model weights only and include all paired prospective training subjects. This run has no honest local holdout.


In [ ]:
RUN_ALL_SUBJECT_CONTINUATION = False  # @param {type:'boolean'}
ALL_SUBJECT_EPOCHS = 10  # @param {type:'integer'}
ALL_SUBJECT_UPDATES_PER_EPOCH = 100  # @param {type:'integer'}
ALL_SUBJECT_DIR = OUTPUT_DIR / 'all_subjects'
assert_runtime_only()
if RUN_ALL_SUBJECT_CONTINUATION:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Acknowledge the completed baseline milestone before training.')
    pilot_checkpoint = PAID_RUN_DIR / 'checkpoints' / 'best.pt'
    if not pilot_checkpoint.is_file():
        raise FileNotFoundError(pilot_checkpoint)
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.train', '--cache-dir', str(PREPROCESSED_DIR), '--output-dir', str(ALL_SUBJECT_DIR), '--preset', 'portable', '--initialize', str(pilot_checkpoint), '--holdout-subject', 'none', '--device', 'cuda', '--amp', 'bf16', '--batch-size', str(PAID_BATCH_SIZE), '--gradient-accumulation', '1', '--crop-size', str(PAID_CROP_SIZE), '--epochs', str(ALL_SUBJECT_EPOCHS), '--updates-per-epoch', str(ALL_SUBJECT_UPDATES_PER_EPOCH), '--workers', str(PAID_WORKERS), '--ema-beta', str(PAID_EMA_BETA), '--left-right-flip-probability', str(PAID_LEFT_RIGHT_FLIP_PROBABILITY), '--loss-lpips', str(PAID_LOSS_LPIPS), '--progress', '--baseline-milestone-complete'])
else:
    print('All-subject continuation is OFF.')


## 11. Select one checkpoint and run full-volume validation smoke

Leave the checkpoint field blank to prefer the all-subject checkpoint, then the paid held-out run, then any checkpoint staged in step 3. The smoke generates exactly one complete 364-slice prediction and validates its metadata and values.


In [ ]:
SELECTED_CHECKPOINT_VALUE = ''  # @param {type:'string'}
RUN_VALIDATION_SMOKE = False  # @param {type:'boolean'}
VALIDATION_SMOKE_DIR = INFERENCE_DIR / 'smoke'
assert_runtime_only()
staged_candidates = sorted(STAGED_CHECKPOINT_DIR.glob('*.pt')) + sorted(STAGED_CHECKPOINT_DIR.glob('*.pth'))
checkpoint_candidates = [Path(SELECTED_CHECKPOINT_VALUE).expanduser()] if SELECTED_CHECKPOINT_VALUE.strip() else [ALL_SUBJECT_DIR / 'checkpoints' / 'best.pt', PAID_RUN_DIR / 'checkpoints' / 'best.pt', *staged_candidates]
SELECTED_CHECKPOINT = next((path for path in checkpoint_candidates if path.is_file()), checkpoint_candidates[0])
print('Selected checkpoint:', SELECTED_CHECKPOINT)
if RUN_VALIDATION_SMOKE:
    if not SELECTED_CHECKPOINT.is_file():
        raise FileNotFoundError(SELECTED_CHECKPOINT)
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.infer', '--checkpoint', str(SELECTED_CHECKPOINT), '--input-root', str(DATA_DIR), '--output-dir', str(VALIDATION_SMOKE_DIR), '--device', 'cuda', '--amp', 'fp16', '--batch-size', '2', '--smoke'])
else:
    print('Validation smoke inference is OFF.')


## 12. Complete resumable 180-volume validation sweep

Public validation targets are private. This validates the prediction structure and numerical properties but does not produce a local leaderboard metric.


In [ ]:
RUN_FULL_VALIDATION = False  # @param {type:'boolean'}
FULL_INFERENCE_DIR = INFERENCE_DIR / 'full'
assert_runtime_only()
if RUN_FULL_VALIDATION:
    if not SELECTED_CHECKPOINT.is_file():
        raise FileNotFoundError(SELECTED_CHECKPOINT)
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.infer', '--checkpoint', str(SELECTED_CHECKPOINT), '--input-root', str(DATA_DIR), '--output-dir', str(FULL_INFERENCE_DIR), '--device', 'cuda', '--amp', 'fp16', '--batch-size', '2', '--resume'])
else:
    print('Full validation inference is OFF.')


## 13. Fetch the current official builder and review live rules

Immediately before packaging, open the two URLs printed below. The notebook records the current organizer Git commit instead of silently using the old prepared pin.


In [ ]:
PREPARE_OFFICIAL_REPO = False  # @param {type:'boolean'}
LIVE_RULES_REVIEWED = False  # @param {type:'boolean'}
LIVE_RULE_URLS = ('https://mrixfields.chihucloud.com/2026/tasks.html', 'https://github.com/MRIxFields/MRIxFields2026/blob/main/Submission/README.md')
assert_runtime_only()
print('Review before packaging:', *LIVE_RULE_URLS, sep='\n - ')
if PREPARE_OFFICIAL_REPO:
    if OFFICIAL_REPO.exists() and not (OFFICIAL_REPO / '.git').is_dir():
        raise RuntimeError(f'Official repo destination exists but is not a Git checkout: {OFFICIAL_REPO}')
    if not (OFFICIAL_REPO / '.git').is_dir():
        subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/MRIxFields/MRIxFields2026.git', str(OFFICIAL_REPO)])
    official_revision = subprocess.check_output(['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
    submission_readme = OFFICIAL_REPO / 'Submission' / 'README.md'
    official_report = {'created_at_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(), 'official_revision': official_revision, 'submission_readme_sha256': hashlib.sha256(submission_readme.read_bytes()).hexdigest(), 'live_rule_urls': list(LIVE_RULE_URLS), 'rules_reviewed': LIVE_RULES_REVIEWED}
    (REPORT_DIR / 'official_builder_provenance.json').write_text(json.dumps(official_report, indent=2) + '\n')
    print(json.dumps(official_report, indent=2))
else:
    print('Official builder preparation is OFF.')


## 14. Build, validate, and zip the experimental Task 3 submission

This calls the current official builder with method `fpsformer`, then checks 60 pair directories, 180 files, no identity mappings, no segmentation content, correct target-field filenames, `(364,436,30)` float32 volumes, finite `[0,1]` values, and a top-level `task3/` archive. It never uploads the archive.


In [ ]:
BUILD_TASK3_ZIP = False  # @param {type:'boolean'}
assert_runtime_only()
if BUILD_TASK3_ZIP:
    if not LIVE_RULES_REVIEWED:
        raise RuntimeError('Review the live Task 3 and submission rules first.')
    builder = OFFICIAL_REPO / 'Submission' / 'build_submission' / 'build_submission.py'
    if not builder.is_file():
        raise FileNotFoundError('Prepare the current official repository first.')
    manifest_path = FULL_INFERENCE_DIR / 'task3_fpsformer_manifest.json'
    manifest = json.loads(manifest_path.read_text())
    if manifest.get('status') != 'complete' or manifest.get('planned_cases') != 180 or len(manifest.get('completed_cases', [])) != 180:
        raise RuntimeError('Full inference manifest is not a complete 180-case run.')
    submission_run_dir = SUBMISSION_DIR / 'fpsformer_best'
    if submission_run_dir.exists():
        raise RuntimeError(f'Use a fresh submission output directory: {submission_run_dir}')
    placeholder = RUNTIME_ROOT / 'unused_task3_seg_placeholder'
    placeholder.mkdir(parents=True, exist_ok=True)
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(OFFICIAL_REPO / 'Baseline') + os.pathsep + environment.get('PYTHONPATH', '')
    subprocess.check_call([sys.executable, str(builder), '--tasks', 'task3', '--predictions-dir', str(FULL_INFERENCE_DIR), '--predictions-seg-dir', str(placeholder), '--output-dir', str(submission_run_dir), '--task3-method', 'fpsformer', '--task3-mode', 'pro_scratch', '--task3-epoch', 'best'], env=environment)
    task_root = submission_run_dir / 'task3'
    modalities = ('T1W', 'T2W', 'T2FLAIR')
    fields = ('0.1T', '1.5T', '3T', '5T', '7T')
    expected_ids = {'0.1T': ('0001', '0002', '0003'), '1.5T': ('0004', '0005', '0008'), '3T': ('0010', '0011', '0012'), '5T': ('0013', '0014', '0015'), '7T': ('0016', '0017', '0018')}
    expected_files = set()
    expected_pred_dirs = set()
    for modality in modalities:
        for source_field in fields:
            for target_field in fields:
                if source_field == target_field:
                    continue
                pred_dir = task_root / modality / f'{source_field}_to_{target_field}' / 'pred'
                expected_pred_dirs.add(pred_dir)
                for subject_id in expected_ids[source_field]:
                    expected_files.add(pred_dir / f'P_{modality}_{target_field}_{subject_id}.nii.gz')
    actual_files = set(task_root.rglob('*.nii.gz'))
    actual_pred_dirs = {path for path in task_root.rglob('pred') if path.is_dir()}
    if actual_files != expected_files or actual_pred_dirs != expected_pred_dirs:
        raise RuntimeError(f'Submission tree mismatch: files={len(actual_files)}/180, pred_dirs={len(actual_pred_dirs)}/60')
    if any('seg' in path.parts or path.name.endswith('_seg.nii.gz') for path in task_root.rglob('*')):
        raise RuntimeError('Task 3 submission contains segmentation content.')
    numerical = []
    for index, path in enumerate(sorted(actual_files), 1):
        image = nib.load(str(path))
        data = image.get_fdata(dtype=np.float32)
        if tuple(image.shape) != (364, 436, 30) or str(image.get_data_dtype()) != 'float32' or not np.isfinite(data).all() or data.min() < -1e-6 or data.max() > 1 + 1e-6:
            raise RuntimeError(f'Invalid slab prediction: {path}')
        numerical.append({'path': str(path.relative_to(submission_run_dir)), 'minimum': float(data.min()), 'maximum': float(data.max())})
        if index % 20 == 0 or index == len(actual_files):
            print(f'Validated {index}/{len(actual_files)} slab predictions')
    archive_path = RESULTS_DIR / 'task3.zip'
    if archive_path.exists():
        raise RuntimeError(f'Archive already exists; preserve it and use a fresh runtime path: {archive_path}')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        archive.write(task_root, 'task3/')
        for path in sorted(task_root.rglob('*')):
            archive.write(path, path.relative_to(submission_run_dir))
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        if not names or not all(name == 'task3/' or name.startswith('task3/') for name in names):
            raise RuntimeError('Archive does not have exactly one top-level task3/ directory.')
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    build_report = {'status': 'ready_for_user_upload', 'archive': str(archive_path), 'archive_size_bytes': archive_path.stat().st_size, 'archive_sha256': digest, 'official_revision': subprocess.check_output(['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True).strip(), 'checkpoint_sha256': manifest['checkpoint_sha256'], 'prediction_files': len(actual_files), 'pair_directories': len(actual_pred_dirs), 'numerical_checks': numerical}
    (REPORT_DIR / 'task3_fpsformer_submission_report.json').write_text(json.dumps(build_report, indent=2) + '\n')
    print(json.dumps({key: value for key, value in build_report.items() if key != 'numerical_checks'}, indent=2))
else:
    print('Task 3 build is OFF.')


## 15. Export only selected final artifacts

This final export cell copies one selected checkpoint plus compact reports/metrics and optionally the validated archive, verifies every copy, and unmounts Drive again. Paid-run recovery backups are handled separately between stopped training chunks. It does not export caches or the full prediction tree.


In [ ]:
EXPORT_SELECTED_ARTIFACTS = False  # @param {type:'boolean'}
EXPORT_NAME_VALUE = 'fpsformer_experiment_01'  # @param {type:'string'}
EXPORT_CHECKPOINT = True  # @param {type:'boolean'}
EXPORT_ARCHIVE = False  # @param {type:'boolean'}
if EXPORT_SELECTED_ARTIFACTS:
    if drive_is_mounted():
        raise RuntimeError('Drive unexpectedly mounted before export.')
    drive.mount(str(DRIVE_MOUNT))
    try:
        export_root = Path(DRIVE_EXPORT_ROOT_VALUE).expanduser() / EXPORT_NAME_VALUE
        if export_root.exists():
            raise RuntimeError(f'Export destination exists; choose a new name: {export_root}')
        export_root.mkdir(parents=True)
        sources = []
        selected = globals().get('SELECTED_CHECKPOINT')
        if EXPORT_CHECKPOINT:
            if selected is None or not Path(selected).is_file():
                raise FileNotFoundError(f'Selected checkpoint is unavailable: {selected}')
            sources.append(Path(selected))
        if EXPORT_ARCHIVE:
            archive_source = RESULTS_DIR / 'task3.zip'
            if not archive_source.is_file():
                raise FileNotFoundError(archive_source)
            sources.append(archive_source)
        for directory in (PAID_RUN_DIR, ALL_SUBJECT_DIR, REPORT_DIR):
            if directory.is_dir():
                sources.extend(path for path in directory.rglob('*') if path.is_file() and (path.name in {'metrics.jsonl', 'provenance.json'} or path.suffix == '.json'))
        seen = set()
        export_records = []
        for source in sources:
            source = source.resolve()
            if source in seen:
                continue
            seen.add(source)
            destination = export_root / source.name
            if destination.exists():
                destination = export_root / f'{source.parent.name}_{source.name}'
            shutil.copy2(source, destination)
            source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
            destination_hash = hashlib.sha256(destination.read_bytes()).hexdigest()
            if source.stat().st_size != destination.stat().st_size or source_hash != destination_hash:
                raise RuntimeError(f'Export verification failed: {destination}')
            export_records.append({'source': str(source), 'destination': str(destination), 'size': source.stat().st_size, 'sha256': source_hash})
        (export_root / 'export_manifest.json').write_text(json.dumps(export_records, indent=2) + '\n')
        print('Verified exports:', json.dumps(export_records, indent=2))
    finally:
        drive.flush_and_unmount()
else:
    print('Artifact export is OFF. Runtime results remain under', RUNTIME_ROOT)
